# 短期记忆

## 1、 基于内存的持久化器

### 1.1 举例1：没有记忆

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

In [7]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[]
)

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
[HumanMessage(content='我叫张三', additional_kwargs={}, response_metadata={}, id='280ffce6-8f94-4c71-8e04-fea9a298f1b9'), AIMessage(content='你好，张三！很高兴认识你。有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 6, 'total_tokens': 20, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '8c8b3aa6-66ad-4744-85e2-afb70b2e8cc5', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19b1c-765b-77c2-b616-e296ba28b7fd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 14, 'total_tokens': 20, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]
Agent

### 1.2 举例2：拥有记忆

In [9]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.messages import HumanMessage

checkpointer = InMemorySaver() #1、创建了内存级的记忆存储

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer #2、让agent具备了存储的能力
)

# 3、同一个thread_id共享记忆的。
config={
    "configurable": {
        "thread_id": "1"
    }
}

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config
)
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 你刚才告诉我你叫**张三**。😊 有什么需要帮忙的吗？


In [10]:
from rich import print as rprint

thread_1_state = agent.get_state(config)

rprint(thread_1_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='d78bcf7a-c221-4860-a36a-97dbb0829cad'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。有什么我可以帮你的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 14,
                        'prompt_tokens': 6,
                        'total_tokens': 20,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 6
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': '98e3b5c7-5f2e-416d-be56-b8337b7e4cfd',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a19b20-d4e9-7ce1-ad58-e0367d49d5ea-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 6,
                    'output_tokens': 14,
                    'total_tokens': 20,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='cc7b7297-94ee-4e52-b49a-1b13f99107c9'
            ),
            AIMessage(
                content='你刚才告诉我你叫**张三**。😊 有什么需要帮忙的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 17,
                        'prompt_tokens': 27,
                        'total_tokens': 44,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 27
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': 'fd418a30-8aa2-4477-840e-248d304d2d02',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a19b20-d801-7933-9e55-d59bea85cc91-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 27,
                    'output_tokens': 17,
                    'total_tokens': 44,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            )
        ]
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f1d68dc-6f93-69f2-8004-c7da498aec98'
        }
    },
    metadata={'source': 'loop', 'st

继续：

In [13]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config
)
print(f"Agent: {response3['messages'][-1].content}")


第三轮对话：
Agent: 你刚才问的是：“**我刚才问了什么问题？**” 😄

这有点像绕口令了，张三。还有什么想聊的吗？


继续：

体会：不同的thread_id不共享记忆

In [15]:
config1 = {
    "configurable": {
        "thread_id": "2"
    }
}
print("\n第四轮对话：")
response4 = agent.invoke({
    "messages": [HumanMessage("我叫什么名字")]},
    config=config1
)
print(f"Agent: {response4['messages'][-1].content}")


第四轮对话：
Agent: 我不知道你叫什么名字。在我们这次对话中，你没有告诉过我你的名字。


In [16]:
thread_2_state = agent.get_state(config1)
rprint(thread_2_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我刚才问了什么问题？',
                additional_kwargs={},
                response_metadata={},
                id='35fe32dc-c2e2-4a20-b957-65cb560e0719'
            ),
            AIMessage(
                content='你刚刚问的问题是：“我刚才问了什么问题？”',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 9,
                        'prompt_tokens': 8,
                        'total_tokens': 17,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 8
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': '723cb207-7646-4afd-9911-bf3ceb9232e3',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a19b24-9f9c-7f23-a59a-c03692e1191c-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 8,
                    'output_tokens': 9,
                    'total_tokens': 17,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='我刚才问了什么问题？',
                additional_kwargs={},
                response_metadata={},
                id='bef84e91-51ca-4037-80bc-4f303fbabf86'
            ),
            AIMessage(
                content='你刚才问的是：“我刚才问了什么问题？”',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 9,
                        'prompt_tokens': 25,
                        'total_tokens': 34,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 25
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': 'b301d92a-6e34-48af-98fd-43cc43424a30',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a19b25-767e-7260-8769-639f7eb5dee0-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 25,
                    'output_tokens': 9,
                    'total_tokens': 34,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='我叫什么名字',
                additional_kwargs={},
                response_metadata={},
                id='7fb02917-5c33-4c5c-865c-bf2861ec92f9'
            ),
            AIMessage(
                content